In [2]:
import pandas as pd
import numpy as np

# Load feature-engineered dataset
df = pd.read_csv(
    "../data/traffic_features.csv",
    parse_dates=["datetime"]
)

# Sort chronologically
df = df.sort_values(
    ["datetime", "GridID"]
).reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())

display(df.head())

Dataset shape: (20880, 19)
Missing values: 0


,GridID,TimeInterval,internet,datetime,local_datetime,hour,date,lag_1,lag_2,lag_3,lag_6,lag_144,rolling_mean_6,rolling_std_6,rolling_mean_144,rolling_std_144,day_of_week,is_weekend,minute_slot
0,1,1383346800000,7.837850,2013-11-01 23:00:00+00:00,2013-11-02 00:00:00+01:00,0,2013-11-02,13.228100,7.957519,9.272482,8.635192,11.028366,9.802627,1.900651,10.465614,3.320239,5,1,0
1,100,1383346800000,5.360477,2013-11-01 23:00:00+00:00,2013-11-02 00:00:00+01:00,0,2013-11-02,5.832877,6.716397,7.479284,7.516680,12.489919,7.017700,0.714691,19.092914,12.620733,5,1,0
2,1000,1383346800000,11.912091,2013-11-01 23:00:00+00:00,2013-11-02 00:00:00+01:00,0,2013-11-02,13.210724,15.245069,16.914793,17.928495,22.413610,16.448760,1.868447,26.554245,18.816914,5,1,0
3,5000,1383346800000,11.141011,2013-11-01 23:00:00+00:00,2013-11-02 00:00:00+01:00,0,2013-11-02,9.784761,12.790233,16.722635,15.875094,14.714089,14.893062,2.993619,13.403071,3.223525,5,1,0
4,10000,1383346800000,14.611517,2013-11-01 23:00:00+00:00,2013-11-02 00:00:00+01:00,0,2013-11-02,15.597499,14.322332,14.874079,18.588053,19.359290,16.560661,2.018960,21.131863,10.099024,5,1,0


In [3]:
# Get unique timestamps in chronological order
timestamps = np.sort(df["datetime"].unique())

# Calculate split indices
train_end = int(len(timestamps) * 0.70)
val_end = int(len(timestamps) * 0.85)

# Split data based on time
train_df = df[df["datetime"].isin(timestamps[:train_end])].copy()
val_df = df[df["datetime"].isin(timestamps[train_end:val_end])].copy()
test_df = df[df["datetime"].isin(timestamps[val_end:])].copy()

print("Training shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining period:",
      train_df["datetime"].min(),
      "to", train_df["datetime"].max())

print("\nValidation period:",
      val_df["datetime"].min(),
      "to", val_df["datetime"].max())

print("\nTesting period:",
      test_df["datetime"].min(),
      "to", test_df["datetime"].max())

Training shape: (14615, 19)
Validation shape: (3130, 19)
Testing shape: (3135, 19)

Training period: 2013-11-01 23:00:00+00:00 to 2013-11-22 06:00:00+00:00

Validation period: 2013-11-22 06:10:00+00:00 to 2013-11-26 14:20:00+00:00

Testing period: 2013-11-26 14:30:00+00:00 to 2013-11-30 22:50:00+00:00


In [4]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Actual internet activity
y_actual = val_df["internet"]

# Predicted activity = previous 10-minute activity
y_pred = val_df["lag_1"]

# Calculate evaluation metrics
mae = mean_absolute_error(y_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_actual, y_pred))
r2 = r2_score(y_actual, y_pred)

print("=== Persistence Baseline (Validation) ===")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

=== Persistence Baseline (Validation) ===
MAE  : 2.7801
RMSE : 4.6417
R²   : 0.8285


In [5]:
# Historical Average Baseline
y_actual = val_df["internet"]

# Predict using average traffic from previous 1 hour
y_pred_avg = val_df["rolling_mean_6"]

# Calculate evaluation metrics
mae_avg = mean_absolute_error(y_actual, y_pred_avg)
rmse_avg = np.sqrt(mean_squared_error(y_actual, y_pred_avg))
r2_avg = r2_score(y_actual, y_pred_avg)

print("=== Historical Average Baseline (Validation) ===")
print(f"MAE  : {mae_avg:.4f}")
print(f"RMSE : {rmse_avg:.4f}")
print(f"R²   : {r2_avg:.4f}")

=== Historical Average Baseline (Validation) ===
MAE  : 2.7633
RMSE : 4.3432
R²   : 0.8499


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Select input features
features = [
    "GridID",
    "hour",
    "day_of_week",
    "is_weekend",
    "minute_slot",
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "lag_144",
    "rolling_mean_6",
    "rolling_std_6",
    "rolling_mean_144",
    "rolling_std_144"
]

# Prepare training and validation data
X_train = train_df[features]
y_train = train_df["internet"]

X_val = val_df[features]
y_val = val_df["internet"]

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict validation traffic
lr_pred = lr_model.predict(X_val)

# Evaluate
lr_mae = mean_absolute_error(y_val, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_val, lr_pred))
lr_r2 = r2_score(y_val, lr_pred)

print("=== Linear Regression (Validation) ===")
print(f"MAE  : {lr_mae:.4f}")
print(f"RMSE : {lr_rmse:.4f}")
print(f"R²   : {lr_r2:.4f}")

=== Linear Regression (Validation) ===
MAE  : 2.4480
RMSE : 3.9332
R²   : 0.8769


In [7]:
from sklearn.ensemble import RandomForestRegressor

# Create Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

# Train model
rf_model.fit(X_train, y_train)

# Predict on validation data
rf_pred = rf_model.predict(X_val)

# Evaluate performance
rf_mae = mean_absolute_error(y_val, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_val, rf_pred))
rf_r2 = r2_score(y_val, rf_pred)

print("=== Random Forest (Validation) ===")
print(f"MAE  : {rf_mae:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"R²   : {rf_r2:.4f}")

=== Random Forest (Validation) ===
MAE  : 2.4688
RMSE : 3.9247
R²   : 0.8774


In [8]:
from xgboost import XGBRegressor

# Create XGBoost model
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# Train model
xgb_model.fit(X_train, y_train)

# Predict validation traffic
xgb_pred = xgb_model.predict(X_val)

# Evaluate performance
xgb_mae = mean_absolute_error(y_val, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_pred))
xgb_r2 = r2_score(y_val, xgb_pred)

print("=== XGBoost (Validation) ===")
print(f"MAE  : {xgb_mae:.4f}")
print(f"RMSE : {xgb_rmse:.4f}")
print(f"R²   : {xgb_r2:.4f}")

ModuleNotFoundError: No module named 'xgboost'

In [9]:
%pip install xgboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.3/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 1.5 MB/s eta 0:00:34
    --------------------------------------- 0.8/48.9 MB 1.8 MB/s eta 0:00:28
   - -------------------------------------- 1.3/48.9 MB 1.6 MB/s eta 0:00:29
   - -------------------------------------- 1.8/48.9 MB 1.9 MB/s eta 0:00:26
   -- ------------------------------------- 2.6/48.9 MB 2.1 MB/s eta 0:00:22
   -- ------------------------------------- 3.4/48.9 MB 2.4 MB/s eta 0:00:20
   --- ------------------------------------ 4.2/48.9 MB 2.7 MB/s eta 0:00:17
   ---- ----------------------------------- 5.5/48.9 MB 3.0 MB/s eta 0:00:15
   ---- ----------------------------------- 6.0/48.9 MB 3.0 MB/s eta 0:00:15
   ----- ---------------------------------- 7.3/48.9 MB 3.2 MB/s eta 0:00:13
   ------ ----------

In [10]:
from xgboost import XGBRegressor

# Create XGBoost model
xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

# Train model
xgb_model.fit(X_train, y_train)

# Predict validation traffic
xgb_pred = xgb_model.predict(X_val)

# Evaluate performance
xgb_mae = mean_absolute_error(y_val, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_val, xgb_pred))
xgb_r2 = r2_score(y_val, xgb_pred)

print("=== XGBoost (Validation) ===")
print(f"MAE  : {xgb_mae:.4f}")
print(f"RMSE : {xgb_rmse:.4f}")
print(f"R²   : {xgb_r2:.4f}")

=== XGBoost (Validation) ===
MAE  : 2.4724
RMSE : 3.9590
R²   : 0.8752


In [11]:
import joblib
import os

# Create models directory if necessary
os.makedirs("../models", exist_ok=True)

# Save trained models
joblib.dump(lr_model, "../models/linear_regression.pkl")
joblib.dump(rf_model, "../models/random_forest.pkl")
joblib.dump(xgb_model, "../models/xgboost.pkl")

print("All three ML models saved successfully!")

All three ML models saved successfully!
